# Week 3, day 2 — Graded 03 SOLUTIONS: Diagnosis   (tier 3 of 4)

Executed in the lab image (pandas 3.0.5) against the real files in `../data/`.
Every quoted number is what it actually printed — including the wrong ones.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Graded 03 — Diagnosis. Run this once.
import numpy as np
import pandas as pd

orders = pd.read_csv("../data/orders_long.csv")
cust = pd.read_csv("../data/customers_messy.csv")

TRUE_TOTAL = round(orders["Sales"].sum(), 2)
print("orders:", orders.shape)
print("the true total Sales, for checking against:", TRUE_TOTAL)

TIER 3 — it ran, and it is wrong

### Question 1

They quoted **`1719.29`**. It is the **mean of 87 orders**. The revenue they meant is **`149578.59`** -- a factor of **87**.

`pivot_table`'s default `aggfunc` is `mean`. The table looks exactly like a
revenue table: same shape, same regions, plausible-looking money.

How you would have caught it: the reconciliation check. Sum the whole table
and compare with `orders["Sales"].sum()`. A table of means will not
reconcile and a table of sums will.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Year", values="Sales")
cell = pt.loc["Ontario", 2012]
sub = orders[(orders["Region"] == "Ontario") & (orders["Year"] == 2012)]["Sales"]

print("what they quoted: %.2f" % cell)
print("what it is:       the MEAN of %d orders" % len(sub))
print("revenue they meant: %.2f" % sub.sum())
print()
print("ratio:", round(sub.sum() / cell, 1), "-- the default aggfunc is mean")

### Question 2

The three bands total **`1331359.98`** against the true `1605576.22` -- **`274216.24` missing, 17.1%**, in **19** unbinned rows.

The top edge is 10,000 and the largest sale is `28389.14`, so 19 orders fell
outside every band and `pd.cut` returned `NaN` for them. `groupby` on the
band then skipped them.

1.7% of rows, 17% of revenue -- the excluded rows are the big ones, which is
always true when you cut on the quantity you are reporting.

How to catch it: `value_counts(dropna=False)` on the band, or compare
`band.notna().sum()` with `len(df)`. Use `float("inf")` as the top edge and
the problem cannot arise.

In [ ]:
band = pd.cut(orders["Sales"], bins=[0, 100, 1000, 10000], labels=["S", "M", "L"])
work = orders.assign(Band=band)
banded = work.groupby("Band", observed=True)["Sales"].sum().sum()

print("sum of the three bands: %.2f" % banded)
print("true total:            %.2f" % TRUE_TOTAL)
print("missing:               %.2f (%.1f%%)"
      % (TRUE_TOTAL - banded, 100 * (TRUE_TOTAL - banded) / TRUE_TOTAL))
print()
print("rows with no band:", int(band.isna().sum()))
print("orders above the top edge:", int((orders["Sales"] > 10000).sum()))
print("max sale: %.2f" % orders["Sales"].max())

### Question 3

The reported mean target `0.2165` is computed over **`487` of `1093` rows** -- **55.4% were dropped**, all of them `Office Supplies`.

`map` returns `NaN` for any value not in the dictionary, and `.mean()`
skips `NaN`. So the 'company-wide' target is the average across the two
categories someone happened to list, and the largest category is absent.

How to catch it: set-difference the values present against the keys covered,
before mapping. One line, and it names the gap.

In [ ]:
target = orders["Category"].map({"Technology": 0.25, "Furniture": 0.15})
print("reported mean target: %.4f" % target.mean())
print()
print("rows with a target:", int(target.notna().sum()), "of", len(orders))
print("rows dropped:      ", int(target.isna().sum()),
      "(%.1f%%)" % (100 * target.isna().sum() / len(orders)))
print()
print("uncovered category:",
      sorted(set(orders["Category"]) - {"Technology", "Furniture"}))

### Question 4

`drop_duplicates()` leaves **`411`** rows but only **`400` distinct CustomerIDs** -- **11** IDs appear twice, differing only by a trailing space.

`drop_duplicates()` compares whole rows for exact equality, and
`'Bill Donatelli'` is not `'Bill Donatelli '`.

How to catch it: compare the row count against `nunique()` on whatever you
believe the key is. If they disagree, your de-duplication has not done what
you think.

The fix is ordering: strip first, then de-duplicate.

In [ ]:
deduped = cust.drop_duplicates()
print("reported as distinct customers:", len(deduped))
print("actual distinct CustomerIDs:   ", deduped["CustomerID"].nunique())
print()
counts = deduped["CustomerID"].value_counts()
repeated = counts[counts > 1]
print("IDs still appearing twice:", len(repeated))
cid = repeated.index[0]
for _, r in deduped[deduped["CustomerID"] == cid].iterrows():
    print("   %s -> %r" % (r["CustomerID"], r["CustomerName"]))

### Question 5

Mean of the 8 regional averages: **`1528.28`**. True average order: **`1468.96`**. A gap of **`59.32`**.

The naive figure weights every region equally, so Nunavut's 9 orders count
as much as Ontario's 302. Nunavut's average is unusually high, and it drags
the unweighted mean up.

This is the mean-of-means trap, and it is everywhere: average of daily
averages, average of per-store margins, average of per-user rates. Whenever
the groups differ in size, the average of the group averages is not the
average.

How to catch it: if you are averaging something that is already an average,
stop and find the underlying rows.

In [ ]:
by_region = orders.groupby("Region")["Sales"].mean()
naive = by_region.mean()
true = orders["Sales"].mean()

print("mean of the 8 regional averages: %.2f" % naive)
print("true average order value:        %.2f" % true)
print("difference:                      %.2f" % (naive - true))
print()
sizes = orders.groupby("Region")["Sales"].agg(["count", "mean"]).round(2)
print(sizes.to_string())
print()
print("-> the naive figure weights Nunavut's %d orders equally with Ontario's %d"
      % (sizes.loc["Nunavut", "count"], sizes.loc["Ontario", "count"]))

### Question 6

The round trip reports **`32`** region-years when only **`30`** have sales -- **2 phantom rows**, both Nunavut (2009 and 2012).

`unstack` made the grid rectangular and invented two cells; `stack` in
pandas 3 kept them as `NaN` rows rather than dropping them.

So a reshape added records to the data. Count them and Nunavut appears to
have Technology history it does not have.

How to catch it: compare the length before and after any reshape round
trip, or `.dropna()` deliberately rather than relying on a default that
changed between pandas versions.

In [ ]:
tech = orders[orders["Category"] == "Technology"]
grouped = tech.groupby(["Region", "Year"])["Sales"].sum()
round_trip = grouped.unstack().stack()

print("reported region-years:", len(round_trip))
print("actually with sales:  ", len(grouped))
print("phantom rows:         ", len(round_trip) - len(grouped))
print()
print("the phantoms:")
print(round_trip[round_trip.isna()].to_string())

### Question 7

The rounded regional totals sum to **`1605576.23`**; the exact total is **`1605576.22`**. A difference of **`0.0125`**.

Each regional total was rounded to the nearest penny for display, and the
errors did not cancel. Summing eight rounded numbers is not the same as
rounding the sum.

A penny is not a problem in itself. It is a problem when a reconciliation
test uses `==`, or when a finance system rejects a batch whose lines do not
add to its header.

**Round for presentation, never for storage.** Keep full precision in
anything a later step consumes.

In [ ]:
rounded = orders.groupby("Region")["Sales"].sum().round(2)
exact = orders["Sales"].sum()

print("sum of the rounded regional totals: %.2f" % rounded.sum())
print("exact company total:                %.2f" % exact)
print("difference:                         %.4f" % (rounded.sum() - exact))
print()
unrounded = orders.groupby("Region")["Sales"].sum()
per_region = (rounded - unrounded).round(4)
print("rounding applied per region:")
print(per_region[per_region != 0].to_string())

### Question 8

The default `groupby` counts **`390`** rows of **`440`** -- **50 silently dropped**, all of them the rows whose `Province` is now `NaN`.

`groupby` discards rows whose key is missing, by default and without
comment. The group sizes are internally consistent and 11% of the file is
absent from them.

It is worse when the missing key is not random: here it is 50 customers you
specifically cleaned, so a report 'by province' silently excludes exactly
the records you were paying attention to.

How to catch it: `dropna=False`, or compare `groupby(...).size().sum()`
with `len(df)`.

In [ ]:
work = cust.copy()
work["Province"] = work["Province"].replace(["-", "?"], np.nan)

default = work.groupby("Province").size()
kept = work.groupby("Province", dropna=False).size()

print("rows counted by the default groupby:", int(default.sum()))
print("rows counted with dropna=False:     ", int(kept.sum()))
print("rows in the frame:                  ", len(work))
print("silently dropped:                   ", len(work) - int(default.sum()))
print()
print("the NaN group:", int(kept[kept.index.isna()].iloc[0]), "rows")

### Question 9

The 'typical order with anomalies excluded' is **`715.08`** against a true mean of **`1468.96`**. It removed **`123` rows (11.3%)** carrying **`911946.60` -- 56.8% of revenue**.

The indefensible number is **56.8%**. A definition of 'anomaly' that
excludes more than half the company's revenue is not describing anomalies,
it is describing the business.

The 1.5xIQR rule is not wrong -- it is doing exactly what it says on a
right-skewed distribution. The error is calling its output anomalous and
reporting the remainder as typical.

How to catch it: always report what an exclusion cost, in rows **and** in
the quantity you care about. The row count alone (11.3%) sounds
reasonable; the revenue share does not.

In [ ]:
s = orders["Sales"]
q1, q3 = s.quantile(0.25), s.quantile(0.75)
iqr = q3 - q1
keep = (s >= q1 - 1.5 * iqr) & (s <= q3 + 1.5 * iqr)

print("reported 'typical order': %.2f" % s[keep].mean())
print("actual mean order:        %.2f" % s.mean())
print()
print("rows removed:    %d (%.1f%%)" % ((~keep).sum(), 100 * (~keep).sum() / len(s)))
print("revenue removed: %.2f (%.1f%%)"
      % (s[~keep].sum(), 100 * s[~keep].sum() / s.sum()))
print()
print("-> the excluded 'anomalies' are %.1f%% of the company's revenue"
      % (100 * s[~keep].sum() / s.sum()))

### Question 10

`cust.pivot(...)` -> **raises** `ValueError: Index contains duplicate entries, cannot reshape`.

It most resembles **Q4**: both are the same underlying fact -- the key you
chose does not identify a row.

The difference is what each API does about it. `pivot()` has to put one
value in one cell, discovers two candidates, and refuses. `drop_duplicates()`
has no such constraint, so it quietly kept both rows and let you report 411
distinct customers.

That is the pattern behind all ten questions. Pandas raises when an
operation is *structurally* impossible and stays silent when it is merely
*wrong*. The nine that ran were all semantically broken and structurally
fine.

Which is why the checks matter more than the exceptions: reconcile totals,
compare counts against `nunique()`, pass `dropna=False`, and report what
every exclusion cost.

In [ ]:
dupes = cust.duplicated(subset=["CustomerID", "Region"]).sum()
print("duplicate CustomerID/Region pairs:", int(dupes))
print(cust.pivot(index="CustomerID", columns="Region", values="Province"))